# River Extraction and Temporal Analysis Workflow

This notebook demonstrates the complete workflow for:
1. Extracting river polygons from binary water/land rasters (from GEE)
2. Processing multiple dates for temporal analysis
3. Detecting channel changes, migration, and braiding

**Author:** Nalaquq LLC / QCORP GIS Training  
**Date:** November 2025  

## Setup

Install required packages (run once):
```bash
pip install -r requirements.txt
```

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from river_extraction import RiverExtractor, batch_extract_rivers
from river_temporal_analysis import RiverTemporalAnalyzer

# Configure visualization
plt.rcParams['figure.figsize'] = (12, 8)
%matplotlib inline

## Part 1: Single Date River Extraction

### Step 1: Load a binary water/land raster from Google Earth Engine

In [ ]:
# Path to your exported GEE raster
# This should be the WaterLand_Classification.tif from the GEE script
raster_path = "path/to/your/WaterLand_Classification.tif"

# Initialize extractor
extractor = RiverExtractor(raster_path)

### Step 2: Preview the binary raster

In [ ]:
# Visualize the input raster
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
im = ax.imshow(extractor.data, cmap='Blues_r')
ax.set_title('Binary Water/Land Raster', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='Value')
plt.tight_layout()
plt.show()

# Show pixel value distribution
unique, counts = np.unique(extractor.data, return_counts=True)
print("\nPixel value distribution:")
for val, count in zip(unique, counts):
    print(f"  Value {val}: {count:,} pixels ({count/extractor.data.size*100:.1f}%)")

### Step 3: Extract river polygons

Two methods available:
- **'largest'**: Extracts the largest connected water body (default)
- **'seed_point'**: Extracts water body containing a specific point (lon, lat)

In [ ]:
# Method 1: Extract largest water body (easiest)
river_gdf = extractor.extract_river(
    method='largest',
    water_value=None,  # Auto-detect
    min_size_pixels=50,  # Remove small noise features
    morphology_iterations=1,  # Light cleaning
    simplify_tolerance=1.0  # Simplify to 1m tolerance
)

# Display results
print("\nExtraction Results:")
print(river_gdf)

In [ ]:
# Method 2: Extract using seed point (if you want a specific water body)
# Uncomment and adjust coordinates as needed

# river_gdf = extractor.extract_river(
#     method='seed_point',
#     seed_point=(-161.904, 59.750),  # (lon, lat) - adjust for your area
#     water_value=None,
#     min_size_pixels=50,
#     morphology_iterations=1,
#     simplify_tolerance=1.0
# )

### Step 4: Visualize extracted river

In [ ]:
# Plot the extracted river polygon(s)
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Plot river polygons
river_gdf.plot(ax=ax, facecolor='blue', edgecolor='darkblue', alpha=0.6, linewidth=1.5)

ax.set_title('Extracted River Polygons (Including Braids)', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

# Print summary statistics
print("\nRiver Summary:")
print(f"  Total area: {river_gdf['area_m2'].sum() / 10000:.2f} hectares")
print(f"  Total perimeter: {river_gdf['perimeter_m'].sum():.2f} meters")
print(f"  Number of polygon features: {len(river_gdf)}")

### Step 5: Save extracted river

In [ ]:
# Save as GeoPackage (recommended) or Shapefile
output_path = "river_extracted_2024-05-01.gpkg"
extractor.save_vector(river_gdf, output_path, driver='GPKG')

print(f"\n✓ Saved to {output_path}")
print("  You can now open this in ArcGIS Pro or QGIS")

## Part 2: Batch Processing (Multiple Dates)

Process multiple binary rasters at once for temporal analysis.

In [ ]:
# List of rasters to process (from different dates)
raster_paths = [
    "path/to/WaterLand_2024-05-01.tif",
    "path/to/WaterLand_2024-06-15.tif",
    "path/to/WaterLand_2024-08-01.tif",
    "path/to/WaterLand_2024-10-30.tif",
]

# Corresponding dates
dates = [
    "2024-05-01",
    "2024-06-15",
    "2024-08-01",
    "2024-10-30"
]

# Output directory for extracted rivers
output_dir = "extracted_rivers"

# Batch process
river_gdfs = batch_extract_rivers(
    raster_paths,
    output_dir,
    method='largest',
    min_size_pixels=50,
    morphology_iterations=1,
    simplify_tolerance=1.0
)

## Part 3: Temporal Analysis

Analyze changes in river geometry over time.

### Step 1: Initialize temporal analyzer

In [ ]:
# Create temporal analyzer
analyzer = RiverTemporalAnalyzer(river_gdfs, dates)

### Step 2: Calculate area changes over time

In [ ]:
# Get area changes
area_df = analyzer.calculate_area_changes()
print("\nRiver Area Over Time:")
print(area_df)

# Plot area changes
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot 1: Absolute area
ax1.plot(area_df['date'], area_df['area_ha'], marker='o', linewidth=2, markersize=8)
ax1.set_ylabel('River Area (hectares)', fontsize=12)
ax1.set_title('River Area Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Plot 2: Percent change
ax2.bar(area_df['date'][1:], area_df['area_change_pct'][1:], color=['green' if x > 0 else 'red' for x in area_df['area_change_pct'][1:]])
ax2.set_ylabel('Area Change (%)', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.set_title('Percent Change in River Area', fontsize=14, fontweight='bold')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Step 3: Detect gains and losses between two dates

In [ ]:
# Compare first and last date
changes = analyzer.detect_gains_and_losses(date_idx1=0, date_idx2=-1)

# Visualize changes
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# Plot stable water (blue)
changes['stable'].plot(ax=ax, facecolor='blue', alpha=0.4, label='Stable')

# Plot gained water (green)
changes['gained'].plot(ax=ax, facecolor='green', alpha=0.7, label='Gained (New Water)')

# Plot lost water (red)
changes['lost'].plot(ax=ax, facecolor='red', alpha=0.7, label='Lost (Abandoned)')

ax.set_title(f'River Changes: {dates[0]} to {dates[-1]}', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

### Step 4: Calculate channel migration distances

In [ ]:
# Calculate migration for each consecutive date pair
migration_stats = []
for i in range(len(dates) - 1):
    stats = analyzer.calculate_migration_distance(i, i + 1, num_samples=200)
    migration_stats.append(stats)

# Plot migration distances
import pandas as pd
migration_df = pd.DataFrame(migration_stats)

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
x = range(len(migration_df))
labels = [f"{migration_df.iloc[i]['date1']} to\n{migration_df.iloc[i]['date2']}" for i in x]

ax.bar(x, migration_df['mean_migration_m'], yerr=migration_df['std_migration_m'], 
       capsize=5, alpha=0.7, label='Mean ± Std')
ax.plot(x, migration_df['max_migration_m'], marker='o', color='red', 
        linewidth=2, markersize=8, label='Maximum')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Migration Distance (meters)', fontsize=12)
ax.set_title('Channel Migration Over Time', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

### Step 5: Detect new channels (avulsion/braiding)

In [ ]:
# Find new channels between first and last date
new_channels = analyzer.detect_new_channels(
    date_idx1=0,
    date_idx2=-1,
    min_area_m2=1000  # Minimum 1000 m² (0.1 hectare)
)

if len(new_channels) > 0:
    print("\nNew Channels Detected:")
    print(new_channels[['channel_area_ha']])
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    river_gdfs[0].plot(ax=ax, facecolor='lightblue', alpha=0.3, label='Original River')
    new_channels.plot(ax=ax, facecolor='green', edgecolor='darkgreen', linewidth=2, alpha=0.8, label='New Channels')
    ax.set_title('New Channels Detected', fontsize=14, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No new channels detected above the minimum size threshold.")

### Step 6: Detect abandoned channels

In [ ]:
# Find abandoned channels
abandoned = analyzer.detect_abandoned_channels(
    date_idx1=0,
    date_idx2=-1,
    min_area_m2=1000
)

if len(abandoned) > 0:
    print("\nAbandoned Channels Detected:")
    print(abandoned[['channel_area_ha']])
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    river_gdfs[-1].plot(ax=ax, facecolor='lightblue', alpha=0.3, label='Current River')
    abandoned.plot(ax=ax, facecolor='red', edgecolor='darkred', linewidth=2, alpha=0.8, label='Abandoned Channels')
    ax.set_title('Abandoned Channels Detected', fontsize=14, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No abandoned channels detected above the minimum size threshold.")

### Step 7: Generate comprehensive report

In [ ]:
# Generate full report with all outputs
report_dir = "temporal_analysis_report"
report = analyzer.generate_report(report_dir)

print("\n" + "="*70)
print("ANALYSIS COMPLETE!")
print("="*70)
print(f"\nAll results saved to: {report_dir}")
print("\nGenerated files:")
print("  - area_changes.csv: Area statistics for each date")
print("  - migration_distances.csv: Channel migration measurements")
print("  - change_map_*.gpkg: Spatial change maps (gains/losses)")
print("  - new_channels_*.gpkg: New channel polygons")
print("  - abandoned_channels_*.gpkg: Abandoned channel polygons")
print("\nYou can open these files in ArcGIS Pro for further analysis.")

## Part 4: Export for ArcGIS Pro / Web App

All outputs are saved as GeoPackage (.gpkg) files which can be directly opened in:
- ArcGIS Pro
- QGIS
- Python (geopandas)
- Web applications (convert to GeoJSON)

### Optional: Convert to GeoJSON for web apps

In [ ]:
# Convert change map to GeoJSON for web display
change_map = report['change_maps'][0]  # First date comparison
change_map.to_file('change_map.geojson', driver='GeoJSON')

print("✓ Saved change_map.geojson for web application")

## Next Steps

1. **Refine parameters**: Adjust `min_size_pixels`, `morphology_iterations`, and `simplify_tolerance` based on your results
2. **Add more dates**: Process additional time periods to build a richer temporal dataset
3. **Field validation**: Compare automated results with ground observations or high-res imagery
4. **Web deployment**: Use these outputs in a Flask app for interactive visualization
5. **ArcGIS Pro analysis**: Import GeoPackages for advanced spatial analysis and mapping

## Questions or Issues?

Contact the QCORP GIS Training team or open an issue in the GitHub repository.